# T11 error analysis — re-run deberta `{1e-5, 5}` with a term dump

E05's cell saved no weights and no predictions: `save_checkpoint` was false, `--save-weights`
was never passed, and the run JSONs hold eight aggregate floats per split. The Colab/Kaggle
sessions are gone, so nothing is recoverable. T11 needs the predictions, so the cell is re-run.

**Only one new artifact is needed.** Recall by term length and recall by gold frequency are set
intersections against the gold key, bucketed by properties of the *gold* term — both computable
locally. So the predicted unique term list is sufficient, and no checkpoint is required.
Exact-span F1 and the invalid-tag counts are already computed inside `evaluate()`.

**Cell 5 runs seed 42 alone** so the reference run lands first and can be inspected before the
other four cost half an hour. **Cell 6 runs seeds 43–46.**

**The numbers will not reproduce E05 exactly.** `cudnn_deterministic` is false and E04 measured
a repeated seed diverging from epoch 2 onward. This is a fresh five-seed sample of the same
config, which doubles as a reproducibility check — report it beside the old cell, do not
silently replace it.

Budget ~40 min: E05 measured this cell at ~440 s per run.

**Two things to set in the right-hand sidebar:**

1. **Session options → Accelerator → GPU T4 x2** (or P100)
2. **Session options → Internet → On** — off by default; the clone and the HuggingFace
   download both fail without it. Needs a phone-verified account.


In [ ]:
# 1. Kaggle guard, internet, GPU, and the command helper.
import os, sys, socket, subprocess, pathlib, shutil, json

if not pathlib.Path('/kaggle').is_dir():
    raise SystemExit('This notebook is for Kaggle.')

try:
    socket.create_connection(('github.com', 443), timeout=10).close()
except OSError as e:
    raise SystemExit(
        f'No internet ({e}). Kaggle disables it by default.\n'
        'Sidebar -> Session options -> Internet -> On (needs a phone-verified account).')

import torch
assert torch.cuda.is_available(), 'No GPU. Sidebar -> Session options -> Accelerator -> GPU.'
print(torch.cuda.get_device_name(0), '|', torch.__version__, '| cuda', torch.version.cuda)

WORK = pathlib.Path('/kaggle/working')        # persisted as notebook Output
REPO = pathlib.Path('/tmp/ate-acter')         # scratch: repo + corpus stay out of Output
GROUP = 't11/deberta_lr1e-05_e5'              # results/runs/<GROUP>/ inside the repo
SRC = REPO / 'results/runs' / GROUP

def run(*args, cwd=None):
    # PYTHONUNBUFFERED: the child's stdout is a pipe, not a tty, so python block-
    # buffers it and nothing appears until ~8 KB accumulates or the process exits.
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    p = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1,
                         cwd=cwd, env=env)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f'exit {p.returncode}: {" ".join(str(a) for a in args)}')

In [ ]:
# 2. Clone the project and the corpus into /tmp.
shutil.rmtree(REPO, ignore_errors=True)
run('git', 'clone', '-q', 'https://github.com/ahmedwaleedaref/ATE-ACTER.git', REPO)
run('git', 'clone', '-q', 'https://github.com/AylaRT/ACTER.git', REPO / 'data/raw/ACTER')
run('git', 'checkout', '-q', 'f05b09e985cad37eeaa8daa8b3f383197aa5324e',
    cwd=REPO / 'data/raw/ACTER')

# Tripwires read the SOURCE, not the commit log: a commit message can say anything,
# and this notebook is useless if the clone predates the T11 wiring -- it would run
# for 40 minutes and write no term list.
for path, needle, why in (
    ('src/models/run_train.py',  '--dump-terms',       'no term list is written, and T11 gets nothing'),
    ('src/models/train_loop.py', 'return_terms',       'same commit as --dump-terms'),
    ('src/eval/spans.py',        'count_invalid_tags', 'the invalid-tag counts will be absent'),
    ('src/models/train_loop.py', 'gold_spans is not None', 'exact-span F1 will be absent'),
):
    assert needle in (REPO / path).read_text(), f'clone predates {needle!r} in {path}: {why}'
assert (REPO / 'data/raw/ACTER/en/htfl/annotated').is_dir(), 'ACTER checkout looks wrong'
print(subprocess.run(['git','log','--oneline','-1'], cwd=REPO,
                     capture_output=True, text=True).stdout)

In [ ]:
# 3. Pinned installs. torch/numpy are Kaggle's -- forcing them breaks its CUDA
#    build. The run JSON records the torch version, so the deviation stays on record.
run(sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==5.16.1', 'tokenizers==0.23.1', 'safetensors==0.8.0',
    'huggingface_hub==1.29.0', 'sentencepiece==0.2.2', 'protobuf==7.36.0',
    'PyYAML==6.0.3', 'pytest==8.3.2')

# seqeval only builds under older setuptools; it is not on the training path.
def _pip(*a):
    return subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *a],
                          capture_output=True, text=True).returncode == 0
HAVE_SEQEVAL = (_pip('--no-build-isolation', 'seqeval==1.2.2')
                or (_pip('setuptools<81', 'wheel')
                    and _pip('--no-build-isolation', 'seqeval==1.2.2')))
print('seqeval:', 'installed (47 tests)' if HAVE_SEQEVAL else
      'UNAVAILABLE -- 46 of 47 tests; training unaffected')

In [ ]:
# 4. Environment check. Any gap from local is a confound to carry into E08.
import importlib
for m in ('torch', 'transformers', 'tokenizers', 'numpy'):
    print(f'{m:14} {importlib.import_module(m).__version__}')
print('python', sys.version.split()[0],
      ' (local: 3.14.4 / torch 2.14.0; E05 ran this cell on Kaggle torch 2.10.0)')
print()
skip = [] if HAVE_SEQEVAL else ['--ignore=tests/test_seqeval_agreement.py']
if skip:
    print('NOTE: seqeval absent -- score_exact_spans is unverified against it here.')
    print('      It IS verified locally, and T11 scores the dumped list locally anyway.')
run(sys.executable, '-m', 'pytest', 'tests/', '-q', *skip, cwd=REPO)

In [ ]:
# 5. Seed 42 alone -- the reference run. Inspect it before spending 30 min on the rest.
def train(seed):
    run(sys.executable, '-m', 'src.models.run_train',
        '--model', 'microsoft/deberta-v3-base',
        '--lr', '1e-5', '--epochs', 5,
        '--group', GROUP, '--seed', seed, '--dump-terms',
        '--reason', 'T11 error analysis: deberta {1e-5,5} re-run with term dump',
        cwd=REPO)
    r = json.loads((SRC / f'seed_{seed}.json').read_text())
    t = r['test']
    # the term file and evaluate() must agree on the type count, or the dump is
    # not the list that produced the score
    assert r['test_term_list']['n_terms'] == t['n_pred_types'], 'term list / n_pred_types mismatch'
    print(f"    SEED {seed}: best_epoch={r['best_epoch']} equi={r['best_equi_f1']:.4f} "
          f"htfl={r['htfl_f1']:.4f} span_f1={t['span_f1']:.4f} types={t['n_pred_types']}")
    print(f"      invalid I: sentence_initial={t['n_i_sentence_initial']} "
          f"after_O={t['n_i_after_o']}  |  {r['wall_time_sec']}s")
    return r

def persist():
    dest = WORK / GROUP
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.rmtree(dest, ignore_errors=True)
    shutil.copytree(SRC, dest)
    print(f'  -> copied to {dest} (persists as notebook Output)')

print('########## seed 42 -- reference run ##########')
train(42)
persist()

# E05's cell, for comparison. A gap here is the nondeterminism E04 measured, not a bug.
print('\nE05 recorded for seed 42: best_epoch=4 equi=0.5657 htfl=0.6039 types=2708')

In [ ]:
# 6. Seeds 43-46. ~30 min. Output is re-copied after each seed, so a dropped
#    session costs one run rather than four.
for s in (43, 44, 45, 46):
    print(f'\n########## seed {s} ##########')
    train(s)
    persist()

In [ ]:
# 7. What to download, and what to do with it.
#
#    Take everything under /kaggle/working/t11/ and drop it into results/runs/t11/
#    locally. Every breakdown is computed locally from terms_seed_*.txt against the
#    gold key -- no GPU, and the harness there is the verified one.
for p in sorted((WORK / 't11').rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(WORK)}  {p.stat().st_size/1024:.0f} KB')

print()
runs = sorted((WORK / GROUP).glob('seed_*.json'))
if len(runs) == 5:
    import statistics
    equi = [json.loads(p.read_text())['best_equi_f1'] for p in runs]
    htfl = [json.loads(p.read_text())['htfl_f1'] for p in runs]
    print(f'equi {statistics.mean(equi):.4f} +/- {statistics.stdev(equi):.4f}  '
          f'(E05: 0.5590 +/- 0.0086)')
    print(f'htfl {statistics.mean(htfl):.4f} +/- {statistics.stdev(htfl):.4f}  '
          f'(E05: 0.5784 +/- 0.0228)')
    print('\nA gap is the nondeterminism E04 measured. Report both cells; do not '
          'overwrite E05.')
else:
    print(f'{len(runs)} of 5 seeds present -- rerun cell 6 for the missing ones.')